In [2]:
"""
combine_checkpoints.py
──────────────────────
Combines all enveloc checkpoint CSVs into one clean file for upload.
Reads from old_checkpoints/ and checkpoints_12_May_2026/ separately,
then merges and deduplicates.
"""
import glob
import numpy as np
import pandas as pd

OLD_CHECKPOINT_DIR = "../data/catalog_output/enveloc_locations/old_checkpoints/"
NEW_CHECKPOINT_DIR = "../data/catalog_output/enveloc_locations/checkpoints_12_May_2026/"
OUTPUT_FILE        = "../data/catalog_output/enveloc_located_all.csv"

MT_RAINIER_LAT, MT_RAINIER_LON = 46.852947, -121.760424

# 1. Load from both directories
old_files = sorted(glob.glob(f"{OLD_CHECKPOINT_DIR}located_events_checkpoint_*.csv"))
new_files = sorted(glob.glob(f"{NEW_CHECKPOINT_DIR}located_events_checkpoint_*.csv"))

print(f"Found {len(old_files)} old checkpoint files...")
print(f"Found {len(new_files)} new checkpoint files (12 May 2026)...")

old_df = pd.concat([pd.read_csv(f) for f in old_files], ignore_index=True) if old_files else pd.DataFrame()
new_df = pd.concat([pd.read_csv(f) for f in new_files], ignore_index=True) if new_files else pd.DataFrame()

df = pd.concat([old_df, new_df], ignore_index=True)

# 2. Deduplicate — new run rows take priority over old (drop_duplicates keeps first,
#    so put new_df first to prefer newer results if the same event_id appears in both)
df = pd.concat([new_df, old_df], ignore_index=True)
df = df.drop_duplicates(subset="event_id", keep="first")
print(f"Total events after dedup : {len(df):,}")

# 3. Keep only successfully located rows
located = df[df["location_status"] == "located"].copy()
print(f"Located events           : {len(located):,}  ({100*len(located)/len(df):.1f}%)")

# 4. Add useful derived columns
located["rounded_start"] = pd.to_datetime(located["rounded_start"], utc=True, format="mixed")
located["year"]  = located["rounded_start"].dt.year
located["month"] = located["rounded_start"].dt.month
located["hour"]  = located["rounded_start"].dt.hour
located["date"]  = located["rounded_start"].dt.date.astype(str)

def haversine_km(lat, lon, ref_lat=MT_RAINIER_LAT, ref_lon=MT_RAINIER_LON):
    R = 6371.0
    dlat = np.radians(lat - ref_lat)
    dlon = np.radians(lon - ref_lon)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(ref_lat)) * np.cos(np.radians(lat)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

located["dist_from_summit_km"] = haversine_km(
    located["enveloc_latitude"], located["enveloc_longitude"]
).round(2)

# 5. Save
cols = [
    "event_id", "rounded_start", "date", "year", "month", "hour",
    "most_common_class", "num_stations", "n_traces_used",
    "enveloc_latitude", "enveloc_longitude", "dist_from_summit_km",
]
located[cols].to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Saved → {OUTPUT_FILE}  ({len(located):,} rows)")
print(f"\nClass breakdown:")
print(located["most_common_class"].value_counts().to_string())
print(f"\nYear range: {located['year'].min()} – {located['year'].max()}")

Found 3254 old checkpoint files...
Found 0 new checkpoint files (12 May 2026)...
Total events after dedup : 325,400
Located events           : 42,672  (13.1%)

✅ Saved → ../data/catalog_output/enveloc_located_all.csv  (42,672 rows)

Class breakdown:
most_common_class
su    35085
px     7587

Year range: 2010 – 2017
